## 1. Install dependencies

In [1]:
!pip -q install transformers datasets accelerate matplotlib tqdm

## 2. Imports and configuration

In [2]:
import os, re, json, html, base64, random, time
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Optional

import torch
import torch.nn as nn
from torch.optim import AdamW
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

# -----------------------------
# Change these for a stronger run
# -----------------------------
MODEL_NAME = "Qwen/Qwen3-0.6B"

# For fast debugging, keep these small.
# For final/bonus experiment, set TRAIN_LIMIT = None and TEST_LIMIT = None.
TRAIN_LIMIT = None
TEST_LIMIT = None

# Coconut paper uses c=2 for GSM8K.
C_LATENTS_PER_STEP = 2

# Stages:
# 0 = full CoT, no latent thoughts
# 1 = remove first CoT step, use 2 latent thoughts
# 2 = remove first 2 CoT steps, use 4 latent thoughts
# 3 = remove first 3 CoT steps, use 6 latent thoughts
# 4 = final stage: remove all CoT, still use 6 latent thoughts, train answer only
STAGES = [0, 1, 2, 3, 4]

# For a serious run, increase these.
EPOCHS_PER_STAGE = {
    0: 1,
    1: 1,
    2: 1,
    3: 1,
    4: 1,
}

LR = 2e-5
MAX_NEW_TOKENS = 80
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
Path("outputs").mkdir(exist_ok=True)

Device: cuda


## 3. Coconut model wrapper

In [3]:
SPECIAL_TOKENS = ["<bot>", "<eot>", "<latent>"]

class CoconutLM(nn.Module):
    """
    Coconut wrapper around a causal LM.

    Core mechanism:
    Every <latent> token embedding is replaced by the previous position's
    final hidden state. This creates continuous hidden-state feedback.
    """

    def __init__(self, model_name=MODEL_NAME):
        super().__init__()
        self.tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        added = self.tokenizer.add_special_tokens({"additional_special_tokens": SPECIAL_TOKENS})

        dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            trust_remote_code=True,
            torch_dtype=dtype,
        )

        if added > 0:
            self.model.resize_token_embeddings(len(self.tokenizer))

        self.bot_id = self.tokenizer.convert_tokens_to_ids("<bot>")
        self.eot_id = self.tokenizer.convert_tokens_to_ids("<eot>")
        self.latent_id = self.tokenizer.convert_tokens_to_ids("<latent>")

    @property
    def dev(self):
        return next(self.parameters()).device

    def tokenize(self, text: str):
        return self.tokenizer(text, return_tensors="pt", add_special_tokens=False)

    def make_latent_inputs_embeds(self, input_ids: torch.Tensor) -> torch.Tensor:
        """
        Starter implementation supports batch size 1.
        This is intentional: it makes the sequential latent pass easy to inspect.
        """
        assert input_ids.shape[0] == 1, "This assignment implementation uses batch size 1."

        embed_layer = self.model.get_input_embeddings()
        inputs_embeds = embed_layer(input_ids).clone()

        latent_positions = (input_ids[0] == self.latent_id).nonzero(as_tuple=True)[0].tolist()

        for pos in latent_positions:
            # Sequential pass:
            # latent[pos] depends on hidden[pos-1], so we run the prefix first.
            prefix_embeds = inputs_embeds[:, :pos, :]
            attention_mask = torch.ones(prefix_embeds.shape[:2], device=prefix_embeds.device, dtype=torch.long)

            out = self.model(
                inputs_embeds=prefix_embeds,
                attention_mask=attention_mask,
                output_hidden_states=True,
                use_cache=False,
            )

            prev_hidden = out.hidden_states[-1][:, -1, :]

            # Important: no detach.
            # This keeps gradients flowing through the latent reasoning chain.
            inputs_embeds[:, pos, :] = prev_hidden

        return inputs_embeds

    def forward(self, input_ids: torch.Tensor, labels: Optional[torch.Tensor] = None):
        input_ids = input_ids.to(self.dev)
        if labels is not None:
            labels = labels.to(self.dev)

        inputs_embeds = self.make_latent_inputs_embeds(input_ids)
        attention_mask = torch.ones(input_ids.shape, device=self.dev, dtype=torch.long)

        return self.model(
            inputs_embeds=inputs_embeds,
            attention_mask=attention_mask,
            labels=labels,
            use_cache=False,
        )

    @torch.no_grad()
    def generate_with_latents(self, prompt: str, k_latents: int, max_new_tokens: int = MAX_NEW_TOKENS):
        self.eval()

        seed = prompt + " <bot>" + (" <latent>" * k_latents) + " <eot>"
        ids = self.tokenize(seed)["input_ids"].to(self.dev)

        for _ in range(max_new_tokens):
            out = self.forward(ids)
            next_id = torch.argmax(out.logits[:, -1, :], dim=-1, keepdim=True)
            ids = torch.cat([ids, next_id], dim=1)

            if next_id.item() == self.tokenizer.eos_token_id:
                break

        text = self.tokenizer.decode(ids[0], skip_special_tokens=False)
        clean = (
            text.replace("<bot>", "")
                .replace("<latent>", "")
                .replace("<eot>", "")
        )
        return text, clean

## 4. Load GSM8K and parse CoT steps

In [4]:
ANSWER_RE = re.compile(r"####\s*(-?\d+(?:\.\d+)?)")

def extract_final_answer(answer_text: str) -> str:
    match = ANSWER_RE.search(answer_text)
    if match:
        return match.group(1).strip()
    nums = re.findall(r"-?\d+(?:\.\d+)?", answer_text)
    return nums[-1].strip() if nums else ""

def parse_gsm8k_steps(answer_text: str) -> List[str]:
    """
    GSM8K answer format usually has reasoning text, then '#### final_answer'.
    We split the reasoning portion into readable steps.
    """
    reasoning = answer_text.split("####")[0].strip()
    reasoning = reasoning.replace("\n", " ")

    # Split on sentence boundaries while keeping simple math expressions intact.
    raw_steps = re.split(r"(?<=[.!?])\s+", reasoning)
    steps = [s.strip() for s in raw_steps if s.strip()]

    if not steps and reasoning:
        steps = [reasoning]

    return steps

def make_record(row):
    return {
        "question": row["question"].strip(),
        "cot_steps": parse_gsm8k_steps(row["answer"]),
        "answer": extract_final_answer(row["answer"]),
        "raw_answer": row["answer"],
    }

gsm = load_dataset("openai/gsm8k", "main")

train_rows = [make_record(x) for x in gsm["train"]]
test_rows = [make_record(x) for x in gsm["test"]]

random.shuffle(train_rows)

if TRAIN_LIMIT is not None:
    train_rows = train_rows[:TRAIN_LIMIT]
if TEST_LIMIT is not None:
    test_rows = test_rows[:TEST_LIMIT]

print("Train examples:", len(train_rows))
print("Test examples:", len(test_rows))
print(json.dumps(train_rows[0], indent=2)[:1200])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/7.93k [00:00<?, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Train examples: 7473
Test examples: 1319
{
  "question": "Stefan goes to a restaurant to eat dinner with his family. They order an appetizer that costs $10 and 4 entrees that are $20 each. If they tip 20% of the total for the waiter, what is the total amount of money that they spend at the restaurant?",
  "cot_steps": [
    "The total cost of the entrees is 4 * $20 = $<<4*20=80>>80.",
    "The total cost of the dinner is $80 + $10 = $<<80+10=90>>90.",
    "The tip is $90 * 0.20 = $<<90*0.20=18>>18 The total cost with tip is $90 + $18 = $<<90+18=108>>108"
  ],
  "answer": "108",
  "raw_answer": "The total cost of the entrees is 4 * $20 = $<<4*20=80>>80.\nThe total cost of the dinner is $80 + $10 = $<<80+10=90>>90.\nThe tip is $90 * 0.20 = $<<90*0.20=18>>18\nThe total cost with tip is $90 + $18 = $<<90+18=108>>108\n#### 108"
}


## 5. Curriculum formatting

In [5]:
def k_latents_for_stage(stage: int) -> int:
    """
    GSM8K Coconut setting:
    stages 1, 2, 3 replace reasoning steps with 2, 4, 6 continuous thoughts.
    final stage keeps 6 continuous thoughts and removes all remaining CoT.
    """
    if stage == 0:
        return 0
    if stage in [1, 2, 3]:
        return stage * C_LATENTS_PER_STEP
    if stage == 4:
        return 3 * C_LATENTS_PER_STEP
    raise ValueError(f"Unknown stage {stage}")

def format_curriculum_example(ex: Dict, stage: int):
    prompt = f"Question: {ex['question']}\nAnswer:"

    if stage == 0:
        target_steps = ex["cot_steps"]
        k_latents = 0
    elif stage in [1, 2, 3]:
        remove_n = min(stage, len(ex["cot_steps"]))
        target_steps = ex["cot_steps"][remove_n:]
        k_latents = k_latents_for_stage(stage)
    elif stage == 4:
        target_steps = []
        k_latents = k_latents_for_stage(stage)
    else:
        raise ValueError(stage)

    latent_block = ""
    if k_latents > 0:
        latent_block = " <bot>" + (" <latent>" * k_latents) + " <eot>"

    if target_steps:
        target = " " + " ".join(target_steps) + f" Therefore, the answer is {ex['answer']}."
    else:
        target = f" The answer is {ex['answer']}."

    full_text = prompt + latent_block + target
    prefix_text = prompt + latent_block

    return {
        "prompt": prompt,
        "prefix_text": prefix_text,
        "full_text": full_text,
        "target": target,
        "k_latents": k_latents,
        "stage": stage,
    }

# Show all curriculum versions for one example
for s in STAGES:
    item = format_curriculum_example(train_rows[0], s)
    print("=" * 80)
    print("STAGE", s, "K_LATENTS", item["k_latents"])
    print(item["full_text"][:1200])

STAGE 0 K_LATENTS 0
Question: Stefan goes to a restaurant to eat dinner with his family. They order an appetizer that costs $10 and 4 entrees that are $20 each. If they tip 20% of the total for the waiter, what is the total amount of money that they spend at the restaurant?
Answer: The total cost of the entrees is 4 * $20 = $<<4*20=80>>80. The total cost of the dinner is $80 + $10 = $<<80+10=90>>90. The tip is $90 * 0.20 = $<<90*0.20=18>>18 The total cost with tip is $90 + $18 = $<<90+18=108>>108 Therefore, the answer is 108.
STAGE 1 K_LATENTS 2
Question: Stefan goes to a restaurant to eat dinner with his family. They order an appetizer that costs $10 and 4 entrees that are $20 each. If they tip 20% of the total for the waiter, what is the total amount of money that they spend at the restaurant?
Answer: <bot> <latent> <latent> <eot> The total cost of the dinner is $80 + $10 = $<<80+10=90>>90. The tip is $90 * 0.20 = $<<90*0.20=18>>18 The total cost with tip is $90 + $18 = $<<90+18=108>

## 6. Tokenize with loss masking

In [6]:
def make_input_and_labels(coconut: CoconutLM, formatted: Dict):
    input_ids = coconut.tokenize(formatted["full_text"])["input_ids"].to(coconut.dev)

    labels = input_ids.clone()
    labels[:] = -100

    prefix_len = coconut.tokenizer(
        formatted["prefix_text"],
        return_tensors="pt",
        add_special_tokens=False
    )["input_ids"].shape[1]

    # Only supervise the remaining reasoning/answer tokens after prompt + latent block.
    labels[:, prefix_len:] = input_ids[:, prefix_len:]

    return input_ids, labels

## 7. Load model

In [7]:
coconut = CoconutLM(MODEL_NAME).to(device)
print("Loaded:", MODEL_NAME)
print("Special IDs:", coconut.bot_id, coconut.latent_id, coconut.eot_id)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded: Qwen/Qwen3-0.6B
Special IDs: 151669 151671 151670


## 8. Curriculum training

In [ ]:
def train_one_stage(coconut: CoconutLM, rows: List[Dict], stage: int, epochs: int, lr: float):
    """
    Trains one curriculum stage.
    Optimizer is intentionally created inside this function, so optimizer state is reset
    when stage changes.
    """
    coconut.train()
    optimizer = AdamW(coconut.parameters(), lr=lr)

    stage_losses = []
    k_latents = k_latents_for_stage(stage)

    print(f"\nStarting stage {stage} | k_latents={k_latents} | epochs={epochs}")

    for epoch in range(epochs):
        random.shuffle(rows)
        pbar = tqdm(rows, desc=f"stage {stage} epoch {epoch+1}/{epochs}")

        for ex in pbar:
            formatted = format_curriculum_example(ex, stage)
            input_ids, labels = make_input_and_labels(coconut, formatted)

            optimizer.zero_grad(set_to_none=True)
            out = coconut(input_ids=input_ids, labels=labels)
            loss = out.loss

            if torch.isnan(loss):
                print("NaN loss skipped")
                continue

            loss.backward()
            torch.nn.utils.clip_grad_norm_(coconut.parameters(), 1.0)
            optimizer.step()

            loss_val = float(loss.detach().cpu())
            stage_losses.append(loss_val)
            pbar.set_postfix(loss=f"{loss_val:.4f}")

    return stage_losses

all_losses = {}
start_time = time.time()

for stage in STAGES:
    losses = train_one_stage(
        coconut=coconut,
        rows=train_rows,
        stage=stage,
        epochs=EPOCHS_PER_STAGE[stage],
        lr=LR,
    )
    all_losses[str(stage)] = losses

    ckpt_dir = Path(f"outputs/checkpoint_stage_{stage}")
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    coconut.model.save_pretrained(ckpt_dir)
    coconut.tokenizer.save_pretrained(ckpt_dir)
    print("Saved", ckpt_dir)

elapsed = time.time() - start_time
print("Training finished in minutes:", elapsed / 60)

Path("outputs/curriculum_losses.json").write_text(json.dumps(all_losses, indent=2), encoding="utf-8")


Starting stage 0 | k_latents=0 | epochs=1


stage 0 epoch 1/1:   0%|          | 0/7473 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved outputs/checkpoint_stage_0

Starting stage 1 | k_latents=2 | epochs=1


stage 1 epoch 1/1:   0%|          | 0/7473 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved outputs/checkpoint_stage_1

Starting stage 2 | k_latents=4 | epochs=1


stage 2 epoch 1/1:   0%|          | 0/7473 [00:00<?, ?it/s]

## 9. Plot curriculum loss curves

In [ ]:
plt.figure(figsize=(8, 5))

offset = 0
for stage in STAGES:
    losses = all_losses[str(stage)]
    xs = list(range(offset, offset + len(losses)))
    plt.plot(xs, losses, label=f"stage {stage}")
    offset += len(losses)

plt.xlabel("Training update")
plt.ylabel("Loss")
plt.title("Coconut curriculum training loss")
plt.legend()
plt.grid(True)
plt.savefig("outputs/curriculum_loss_curve.png", dpi=160, bbox_inches="tight")
plt.show()